In [2]:
def trs_setup():
    field_list = ['soy1', 'soy2', 'soy3']#, 'soy4', 'soy5', 'soy6', 'corn1', 'corn2', 'corn3', 'corn4', 'corn5', 'corn6']
    soy1 = [
        ['0111','2024-05-11-00-00']
    ]
    soy2 = [
        ['0113','2024-05-11-00-00']
    ]
    soy3 = [
        ['0115','2024-05-11-00-00']
    ]

    '''
    soy4 = [
        ['0117','2024-05-11-00-00']
    ]
    soy5 = [
        ['0119','2024-05-11-00-00']
    ]
    soy6 = [
        ['0121','2024-05-11-00-00']
    ]

    corn1 = [
        ['0111','2024-05-11-00-00']
    ]
    corn2 = [
        ['0113','2024-05-11-00-00']
    ]
    corn3 = [
        ['0115','2024-05-11-00-00']
    ]
    corn4 = [
        ['0117','2024-05-11-00-00']
    ]
    corn5 = [
        ['0119','2024-05-11-00-00']
    ]
    corn6 = [
        ['0121','2024-05-11-00-00']
    ]
    '''
    return field_list, [soy1, soy2, soy3,]# soy4, soy5, soy6, corn1, corn2, corn3, corn4, corn5, corn6]

In [8]:
# // this part is download the data from the website for 2 month.
import requests
import sys
import os
import json
import pandas as pd
from pprint import pprint
import datetime
import pytz
# import config  # I don't know what is this config but this is meaningless DK 2024-06-24
from dateutil.relativedelta import relativedelta

# ! set up the emergence date for the soybean/corn anything early
emergence_date = datetime.datetime(2024, 5, 11)
deviceList = []

# ** set the configuration for the request                                                                                  **
config = {
 'username' : 'yang2309@purdue.edu', ### Insert your email address used by AgIT Thingsboard system
 'password': 'dsya2002',  ### Insert your AgIT thingsboard password
 'server' : 'https://things.iot.ag.purdue.edu:8080'
}

# ** defining the function to get the token for the request and setting the header for the request                          **
def getCustomerDevices(custID, textSearch=None):
    parameters = {        
        'pageSize': 1000,
        'page': 0,                
    }
    att_parms = {
        'keys': 'dev_eui'
    }
    if(textSearch):
        parameters.update({'textSearch': textSearch})
    responseList = requests.get(f"{config['server']}/api/customer/{custID}/devices", headers=TBheaders,params= parameters).json()
    #pprint(responseList)
    list = []
    for dev in responseList['data']:
        #pprint(dev)
        #print('------------------------------------------------------------------------------------------')
        #'id': {'entityType': 'DEVICE', 'id': 'd49153a0-c868-11eb-95d8-09d06ef6a9a5'},
        url = f"{config['server']}/api/plugins/telemetry/DEVICE/{dev['id']['id']}/values/attributes"
        deviceResp = requests.get(url, headers=TBheaders,params= att_parms).json()
        #print('------------------------------------------------------------------------------------------')
        list.append([dev['id']['id'],dev['name'],deviceResp[0]['value']])
    return list
        
def login(url, username, password):
    # Log into ThingsBoard
    return requests.post(f"{url}/api/auth/login", json={
        "username": username,
        "password": password
    }).json()['token']

def get_keys(device):
    return requests.get(f"{config['server']}/api/plugins/telemetry/DEVICE/{device}/keys/timeseries",
                 headers=TBheaders).json()

def get_data_chunk(url, token, device, key, start, stop, limit):
    #print([url, device, key, start, stop, limit])
    return requests.get(f"{url}/api/plugins/telemetry/DEVICE/{device}/values/timeseries",
             headers=TBheaders,
            params= {
                'keys': key,
                'startTs': start,
                'endTs': stop,
                'limit': limit,
                'agg': 'NONE'
            }).json()

def get_data(url, token, device, key, start, stop):
    global totalLength
    p = pd.DataFrame()
    
    # You have to request data backwards in time ...
    while start < stop:
        data = get_data_chunk(url, token, device[0], key, start, stop, 100000)
        #print(data)
        if key not in data:
            break;
        
        #print(f"{key}: Loaded {len(data[key])} points")
        t = pd.DataFrame.from_records(data[key])
        #t['Timestamp'] = t['ts']
        #pprint(t['ts'])
        t['ts'] = (pd.to_datetime(t['ts'],unit='ms'))
        # ! substract 4 hours from the ts
        t['ts'] = t['ts'] - pd.Timedelta(hours=4)       
        t.set_index('ts', inplace=True)
        
        t.rename(columns={'value': key}, inplace=True)
        p = p._append(t)

        # Update "new" stop time
        stop = data[key][-1]['ts'] - 1
    totalLength += len(p)
    #print(f"Total Length: {totalLength}")
    return p

def outputCSV(devices):
    global totalLength
    final_df = pd.DataFrame()
    for device in devices:
        #print(f"Downloading DEVICE: {device[0]} data");
        #print(device)
        p = pd.DataFrame()
        for key in keys:
            #print(f"info: Pulling {key}...");
            tempin = get_data(config['server'], token, device, key, startTS, endTS)            
            if(len(tempin)>0):                
                p = pd.concat([p,tempin], axis=1)
        p['Entity Name'] = device[1]
        p['dev_eui'] = device[2]
        p.reset_index(drop=False)
        #p_new_index = p.assign(**{'Timestamp': p.index})        
        if(len(p)):
            final_df = pd.concat([final_df,p])
        
    # Create Time Strings
    # Convert to nanoseconds for pandas.to_datetime
    start_timestamp_ns = startTS * 1000000
    end_timestamp_ns = endTS * 1000000
    
    # Convert timestamp to datetime object
    start_dt = pd.to_datetime(start_timestamp_ns, unit='ns')
    end_dt = pd.to_datetime(end_timestamp_ns, unit='ns')
    
    # Format datetime string as yyyy-mm-dd-HH-MM
    start_formatted_string = start_dt.strftime('%Y-%m-%d-%H-%M')
    end_formatted_string = end_dt.strftime('%Y-%m-%d-%H-%M')
    # Select variables to export
    df_order = ["Entity Name","data_soil_moisture1","data_soil_moisture2","data_soil_moisture3","data_soil_moisture4","data_tem1","data_tem2","data_tem3","data_tem4","data_tem5","data_tem6","data_tem7","dev_eui"]
    final_df = final_df.reindex(columns=df_order)
    final_df1 = final_df.sort_values(by='ts')
    # replace certain values in the 'Entity Name' column
    final_df1['Entity Name'] = final_df1['Entity Name'].str.replace({'ABE-DRAGINO-GROPOINT-CHERKHAUER-ACRE-',''})
    
    # Get current time
    now = datetime.datetime.now()
    
    # Format time string (hours and minutes)
    formatted_time = now.strftime("%H-%M")
    final_df1.to_csv(f"./99_Raw_data/data-{end_formatted_string}.csv")
    print("File Export Done.")

def getDeviceCredentialsByDeviceId(deviceID = 0):
    url = config['server']+'/device/'+deviceID+'/credentials'
    resp = requests.get(url,headers=TBheaders)
    responseList = resp.json()
    #pprint(responseList)
    return responseList['credentialsID']

def getDeviceServerAttributes(deviceID = 0):
    if deviceID == 0:
        while(deviceID == 0):
            try:
                deviceID = input("Enter device ID: ")
            except:
                print("Invalid DeviceID")
    url = config['server']+'/plugins/telemetry/DEVICE/'+deviceID+'/values/attributes'
    #pprint(url)
    #pprint(TBheaders)
    xresp = requests.get(url,headers=TBheaders)
    #pprint(xresp)
    #pprint(resp.content())
    #print(xresp.text())
    responseList = xresp.json()
    #pprint(responseList)
    #return responseList['credentialsID']


# ** getting token for the request                                                                                         **
print("Server: ",config['server'])
token = login(config['server'], config['username'], config['password']);
print(f"Token: {token}")
TBheaders={ 'Accept': '*/*', 'X-Authorization': f"Bearer {token}" }

# Create a datetime object representing the local date and time
# Year, Month, Day, Hour, Minute
today_dt = datetime.datetime.now()
start_dt = datetime.datetime(emergence_date.year, emergence_date.month, emergence_date.day, 18, 0)
end_dt = datetime.datetime(today_dt.year, today_dt.month, today_dt.day, 6, 00)
print (start_dt, end_dt)

# Convert to a specific time zone (e.g., UTC)
start_tz_utc = pytz.timezone("UTC")
start_dt_utc = start_tz_utc.localize(start_dt)
end_tz_utc = pytz.timezone("UTC")
end_dt_utc = end_tz_utc.localize(end_dt)

# Extract the Unix timestamp
startTS = int(start_dt_utc.timestamp())*1000
endTS = int(end_dt_utc.timestamp())*1000

# ** customer ID for the request                                                                                            **
devices = getCustomerDevices("7576b020-ecae-11ec-b72b-5dd76ca52a2b","ABE-DRAGINO-GROPOINT-CHERKHAUER-ACRE")
# pprint(devices)

totalLength = 0
keys = ["data_soil_moisture1","data_soil_moisture2","data_soil_moisture3","data_soil_moisture4","data_tem1","data_tem2","data_tem3","data_tem4","data_tem5","data_tem6","data_tem7"]


end_dt = datetime.datetime(today_dt.year, today_dt.month, today_dt.day, 6, 00)
end_formatted_string = end_dt.strftime('%Y-%m-%d-%H-%M')
if f"data-{end_formatted_string}.csv" in os.listdir("./99_Raw_data"):
    print("S-M File Already Exists.")
else:
    print('Downloading Data...')
    outputCSV(devices)

Server:  https://things.iot.ag.purdue.edu:8080
Token: eyJhbGciOiJIUzUxMiJ9.eyJzdWIiOiJ5YW5nMjMwOUBwdXJkdWUuZWR1IiwidXNlcklkIjoiNjRlOWZjYjAtZjc0ZS0xMWVlLWIzYmMtN2ZlNjliZjhkNDExIiwic2NvcGVzIjpbIkNVU1RPTUVSX1VTRVIiXSwic2Vzc2lvbklkIjoiMWIzZWQ2YzMtY2MyMS00ZGVhLTg3ZWUtYTY5MjAzNzM1N2FjIiwiaXNzIjoidGhpbmdzYm9hcmQuaW8iLCJpYXQiOjE3MjU0MTc4MDQsImV4cCI6MTcyNTQyNjgwNCwiZmlyc3ROYW1lIjoiRG9uZ3Nlb2siLCJsYXN0TmFtZSI6IllhbmciLCJlbmFibGVkIjp0cnVlLCJpc1B1YmxpYyI6ZmFsc2UsInRlbmFudElkIjoiYWFjNjU1YTAtYWM2Mi0xMWVjLWFiYzgtMWYxYzA5NTgwZTY3IiwiY3VzdG9tZXJJZCI6Ijc1NzZiMDIwLWVjYWUtMTFlYy1iNzJiLTVkZDc2Y2E1MmEyYiJ9.5vLkl6SUu7Nuas2LT1gCf1kg0HAloc25TeV_VU-93I66kGcz_IBb57qStGofOtv2ln9xGg9Uu0uZdFVjuyOQsQ
2024-05-11 18:00:00 2024-09-03 06:00:00
S-M File Already Exists.


## get_sm_data, merge_sm_data

### description
this block gets the trs info from trs_setup function and extract data from each trs and soil moisture data.
at the end, it will merge data so you can have whole soil moisture data of the one plot.

In [10]:
def get_sm_data(trs_segmenet):
    try:
        end_dt = datetime.datetime.strptime(trs_segmenet[2], '%Y-%m-%d-%H-%M')
    except:
        end_dt = datetime.date.today()
        # set hour and minute to 6:00
        end_dt = datetime.datetime(end_dt.year, end_dt.month, end_dt.day, 6, 0)

    trs = trs_segmenet[0]
    start_dt = datetime.datetime.strptime(trs_segmenet[1], '%Y-%m-%d-%H-%M')
    print(trs, '/', start_dt, '/', end_dt)

    # filter raw_data using the trs as 'Entity Name', within the start_dt and end_dt
    temp = raw_data[(raw_data['Entity Name'] == int(trs)) & (raw_data['ts'] >= start_dt) & (raw_data['ts'] <= end_dt)]
    # set the index to 'ts'
    temp.set_index('ts', inplace=True) 
    # set non numeric values to NaN
    temp = temp.apply(pd.to_numeric, errors='coerce')
    return temp
    

In [31]:
def merge_sm_data(field_name, trs_info):
    sm_data = pd.DataFrame()
    for segment in trs_info:
        sm_temp = get_sm_data(segment)
        sm_data = pd.concat([sm_data, sm_temp])

    sm_data['data_soil_moisture1'] = sm_data['data_soil_moisture1'].astype(float)
    sm_data['data_soil_moisture2'] = sm_data['data_soil_moisture2'].astype(float)
    sm_data['data_soil_moisture3'] = sm_data['data_soil_moisture3'].astype(float)
    sm_data['data_soil_moisture4'] = sm_data['data_soil_moisture4'].astype(float)

    sm_data.to_csv(f'./99_Raw_data/merged_sm_{field_name}_data.csv')
    
    return sm_data

## Bumpfinder

In [29]:
def bumpfinder(sm_data, field_name, timewindow, threshold, target_layer):
    # find the bump in the sm_data
    # timewindow is the time window to search for the bump (hrs)
    # threshold is the threshold for the bump
    # target_layer is the layer to search for the bump
    target_layer = 'data_soil_moisture'+str(target_layer)
    # return the bump data
    bump_data = pd.DataFrame()

    # from top to the bottom of the data frame, find the bump using the timewindow and threshold
    # rows to search will be 2*timewindow
    datawindow = 2*timewindow
    # for each row in the sm_data, find the bump
    # save the 2*timewindow rows as temp and find the maximum and minimum value in the temp
    for i in range(len(sm_data)-datawindow):
        temp = sm_data.iloc[i:i+datawindow]
        # find the maximum value and the minimum value in the timewindow, and if it is greater than the threshold, then it is a bump
        max_value = temp[target_layer].max()
        min_value = temp[target_layer].min()
        if max_value - min_value > threshold:
            # if the bump is found, then find the date of the bump
            # find the date of the maximum value in the temp
            # find the row with the highest value in the day(00:00-24:00) of the max_date
            max_date = temp[temp[target_layer] == max_value].index[0]
            max_date = max_date.replace(hour=0, minute=0)
            temp2 = sm_data.loc[max_date:max_date+datetime.timedelta(days=1)]
            max_value = temp2[target_layer].max()
            # if the max_value is found, then save the row of the maximum value to the bump_data
            # if there are multiple rows with the same max_value, then save first the rows to the bump_data
            bump_data = bump_data.append(temp2[temp2[target_layer] == max_value].iloc[0])

    # remove the duplicate rows
    bump_data = bump_data.drop_duplicates()
    # save the bump_data to the csv file
    bump_data.to_csv(f'./99_Raw_data/bump_data_{field_name}.csv')
            
    return bump_data


## fcfinder

In [68]:
def fcfinder(field_name, target_layer, threshold=0.3):
    target_layer = 'data_soil_moisture'+str(target_layer)

    # find the field capacity after the bump
    # return the fc_data
    fc_data = pd.DataFrame()
    # for each bump date, find the first flat line after the bump from the field_sm_data
    # get the bump date from the bump_data
    bump_dates = bump_data.index
    # for each in bump_dates, get the data after the bump_date from the field_sm_data of the target_layer column.
    # search window is +-4hours from the midnight of the bump_date
    # compare min and max of the temp, if the difference is less than 0.01, then it is the add data to the fc_data
    for i in bump_dates:
        fc_temp = pd.DataFrame()
        for r in range(14):
            r = r+1
            i = i.replace(hour=0, minute=0)
            # get the data from the field_sm_data using search window
            temp = field_sm_data.loc[i+datetime.timedelta(days=r)-datetime.timedelta(hours=4):i+datetime.timedelta(days=r)+datetime.timedelta(hours=4)]
            # print(temp)
            if temp[target_layer].max() - temp[target_layer].min() < threshold:
                # if the difference is less than 0.01, save the row of minimum value to the fc_data
                min_value = temp[target_layer].min()
                fc_temp = fc_temp.append(temp[temp[target_layer] == min_value].iloc[0])
                # print(i, r)
        try:
            fc_data = fc_data.append(fc_temp.iloc[0])
        except:
            pass
    # save the fc_data to the csv file
    fc_data.to_csv(f'./99_Raw_data/fc_data_{field_name}.csv')


In [69]:
# implementation block
import pandas as pd
# ignore warning
import warnings
warnings.filterwarnings('ignore')

field_list, trs_info = trs_setup()
raw_data = pd.read_csv(f"99_Raw_data/data-{end_formatted_string}.csv", parse_dates=['ts'])
print(raw_data.columns)
raw_data.drop(columns=['data_tem1', 'data_tem2', 'data_tem3', 'data_tem4', 'data_tem5', 'data_tem6', 'data_tem7', 'dev_eui'], inplace=True)
# print(raw_data.describe())
# print(raw_data.info())

for i in range(len(trs_info)):
    print(field_list[i])
    # collect the soil moisture data from each field using trs record :: trs_setup()
    field_sm_data = merge_sm_data(field_list[i], trs_info[i])
    # change column type to float
    # print (field_sm_data.info())

    # from field_sm_data, find the bump and drop in the soil moisture data
    bump_data = bumpfinder(field_sm_data, field_list[i], 6, 10, 2)

    fcfinder(field_list[i], 2, threshold=0.5)


    




Index(['ts', 'Entity Name', 'data_soil_moisture1', 'data_soil_moisture2',
       'data_soil_moisture3', 'data_soil_moisture4', 'data_tem1', 'data_tem2',
       'data_tem3', 'data_tem4', 'data_tem5', 'data_tem6', 'data_tem7',
       'dev_eui'],
      dtype='object')
soy1
0111 / 2024-05-11 00:00:00 / 2024-09-03 06:00:00
soy2
0113 / 2024-05-11 00:00:00 / 2024-09-03 06:00:00
soy3
0115 / 2024-05-11 00:00:00 / 2024-09-03 06:00:00
